In [42]:
import numpy as np
import matplotlib.pyplot as plt

In [43]:
thetas = [0, 0, 4, 4]
sigmas = [1, 2, 1, 2]
n = 50
m = 10**4 

distributions = [
    ("Normal", lambda theta, sigma, n: np.random.normal(loc=theta, scale=sigma, size=n)),
    ("Logistic", lambda theta, sigma, n: np.random.logistic(loc=theta, scale=sigma, size=n)),
    ("Cauchy", lambda theta, sigma, n: np.random.standard_cauchy(size=n) * sigma + theta)
]

In [44]:
print(distributions[0][0])

Normal


In [45]:
def est_1(x):
    return np.mean(x)

In [46]:
def est_2(x):
    return np.median(x)

In [47]:
estimators = [est_1, est_2]

In [48]:
def theta_stats(estimates, true_theta, true_sigma):
    mean_est = np.mean(estimates) 
    var_est = np.var(estimates, ddof=1)
    bias = mean_est - true_theta
    mse = np.mean((estimates - true_theta)**2)
    return mean_est, var_est, bias, mse

In [51]:
import os
import pandas as pd

os.makedirs("plots", exist_ok=True)

for theta, sigma in zip(thetas, sigmas):
    for name, generator in distributions:
        estimates = {f"est_{i+1}": [] for i in range(len(estimators))}
        for _ in range(m):
            data = generator(theta, sigma, n)
            for i, est in enumerate(estimators):
                estimates[f"est_{i+1}"].append(est(data))

        est_values = [np.array(estimates[f"est_{i+1}"]) for i in range(len(estimators))]

        est_stats = [theta_stats(estimates, theta, sigma) for estimates in est_values]

        plt.figure(figsize=(8,6))
        plt.boxplot(est_values, tick_labels=[f"est_{i+1}" for i in range(len(estimators))])
        plt.title(f"{name} distribution (theta={theta}, sigma={sigma})")
        plt.ylabel("Wartości estymatorów")
        plt.grid(True)
        
        all_data = np.hstack(est_values)        # wszystkie dane razem
        q1 = np.percentile(all_data, 25)
        q3 = np.percentile(all_data, 75)
        iqr = q3 - q1

        # delta określa, jak bardzo chcemy rozszerzyć zakres ponad standardowy 1.5*IQR
        delta = 5
        plt.ylim(q1 - 1.5*iqr - delta, q3 + 1.5*iqr + delta)

        # zapis PNG
        png_filename = f"plots/{name}_boxplot_theta{theta}_sigma{sigma}.png"
        plt.savefig(png_filename)
        plt.close()

        # Statystyki w pandas
        stats_dict = {
            "Estimator": [],
            "Mean": [],
            "Variance": [],
            "Bias": [],
            "MSE": []
        }

        for i, est in enumerate(estimators):
            mean_est, var_est, bias, mse = est_stats[i]
            stats_dict["Estimator"].append(f"est_{i+1}")
            stats_dict["Mean"].append(mean_est)
            stats_dict["Variance"].append(var_est)
            stats_dict["Bias"].append(bias)
            stats_dict["MSE"].append(mse)

        df_stats = pd.DataFrame(stats_dict)

        # zaokrąglamy do 2 miejsc po przecinku
        df_stats = df_stats.round(2)

        # zapis do CSV przez pandas
        csv_filename = f"plots/{name}_stats_theta{theta}_sigma{sigma}.csv"
        df_stats.to_csv(csv_filename, index=False)

        print(f"Zapisano wykres: {png_filename} i statystyki: {csv_filename}")


Zapisano wykres: plots/Normal_boxplot_theta0_sigma1.png i statystyki: plots/Normal_stats_theta0_sigma1.csv
Zapisano wykres: plots/Logistic_boxplot_theta0_sigma1.png i statystyki: plots/Logistic_stats_theta0_sigma1.csv
Zapisano wykres: plots/Cauchy_boxplot_theta0_sigma1.png i statystyki: plots/Cauchy_stats_theta0_sigma1.csv
Zapisano wykres: plots/Normal_boxplot_theta0_sigma2.png i statystyki: plots/Normal_stats_theta0_sigma2.csv
Zapisano wykres: plots/Logistic_boxplot_theta0_sigma2.png i statystyki: plots/Logistic_stats_theta0_sigma2.csv
Zapisano wykres: plots/Cauchy_boxplot_theta0_sigma2.png i statystyki: plots/Cauchy_stats_theta0_sigma2.csv
Zapisano wykres: plots/Normal_boxplot_theta4_sigma1.png i statystyki: plots/Normal_stats_theta4_sigma1.csv
Zapisano wykres: plots/Logistic_boxplot_theta4_sigma1.png i statystyki: plots/Logistic_stats_theta4_sigma1.csv
Zapisano wykres: plots/Cauchy_boxplot_theta4_sigma1.png i statystyki: plots/Cauchy_stats_theta4_sigma1.csv
Zapisano wykres: plots/No